In [1]:
import networkx as nx
import pandas as pd
import torch
from torch_geometric.data import HeteroData
import pickle
from collections import defaultdict
import re
from tqdm import tqdm
import numpy as np
from matplotlib_venn import venn2
import ast

/home/xyu/.conda/envs/firegnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_graph(filepath):
    with open(filepath, 'rb') as f:
        G = pickle.load(f)
    print(f"Load graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

In [3]:
def get_common_proteins(sample_protein_dict:dict):
    
    disease_proteins = {}
    for pr_list in sample_protein_dict.values():
        for pr in pr_list:
            if pr not in disease_proteins:
                disease_proteins[pr]=1
            else:
                disease_proteins[pr]+=1

    disease_proteins = dict(sorted(disease_proteins.items(), key=lambda x: x[1], reverse=True))

    return disease_proteins

In [4]:
def paprnetwork_to_hetero_data(G: nx.MultiDiGraph, topo_features=True, label_homophily=True):
    """
    Converts the Patient-Protein network into a PyTorch Geometric HeteroData object.
    .
    """
    print("Starting conversion from NetworkX to HeteroData...")
    data = HeteroData()
    node_mappings = {} # {node_type: {node_name: integer_index}}
    
    # 1. Categorize Nodes and Map Indices
    for node, attrs in G.nodes(data=True):
        n_type = attrs.get('type') or attrs.get('label')
        
        if n_type not in node_mappings:
            node_mappings[n_type] = {}
        
        if node not in node_mappings[n_type]:
            node_mappings[n_type][node] = len(node_mappings[n_type])
   
    # 2. Process ALL Node Types (Initialize x)
    for n_type, mapping in node_mappings.items():

        num_nodes = len(mapping)
        data[n_type].num_nodes = num_nodes

        # Initialize x-feature for all node types (Patient & Protein)
        n_ids = sorted(mapping, key=mapping.get)
        x = torch.tensor([G.nodes[nid]['x'] for nid in n_ids], dtype=torch.float)
        data[n_type].x = x
        data[n_type].y = torch.tensor([G.nodes[nid]['y'] for nid in n_ids], dtype=torch.long)

        if topo_features:
            nt_topo_features = compute_topological_features(G, node_type=n_type,label_homophily=label_homophily)

            nt_topo = np.vstack([nt_topo_features[name] for name in nt_topo_features.keys()]).T
            data[n_type].topo_features = torch.tensor(nt_topo,dtype=torch.float)

        
    # 3. Process Edges
    edges_dict = {}
    for u, v, r, attrs in G.edges(keys=True, data=True):
        u_type = G.nodes[u].get('type') or G.nodes[u].get('label')
        v_type = G.nodes[v].get('type') or G.nodes[v].get('label')
        if not isinstance(r, str):
            rel = attrs.get('relation') or attrs.get('rel') or attrs.get('type')
        else:
            rel = r
        
        # Replace double underscores with single ones to satisfy PyG requirements
        safe_rel = str(rel).replace('__', '_')
        
        edge_type = (u_type, safe_rel, v_type)
            
        if edge_type not in edges_dict:
            edges_dict[edge_type] = []
        
        u_idx = node_mappings[u_type][u]
        v_idx = node_mappings[v_type][v]
        edges_dict[edge_type].append([u_idx, v_idx])

    # Finalize Edges in HeteroData
    for etype, content in edges_dict.items():
        data[etype].edge_index = torch.tensor(content, dtype=torch.long).t().contiguous()
    
    print(f"HeteroData created: {len(data.node_types)} node types, {len(data.edge_types)} edge types.")
    
    return data, node_mappings

In [5]:
G = load_graph("../../EdgeAssignmentTask/gumbel_softmax/results/no_label_homophily/adni/AD_KG/ecdf_5/G_adni_Protein.pkl")

data, _ = paprnetwork_to_hetero_data(G,False, False)

Load graph with 1620 nodes and 132704 edges
Starting conversion from NetworkX to HeteroData...


/tmp/ipykernel_851271/2658695397.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  x = torch.tensor([G.nodes[nid]['x'] for nid in n_ids], dtype=torch.float)


HeteroData created: 2 node types, 8 edge types.


### Compute topo features

* **Differential Connectivity Score (DCS):** For each protein $p$ connected to sample $s$, look up its frequency in the disease group ($F_D(p))$ versus the control group ($F_C(p)$). Compute a net score for the sample:

$$
DCS(s) = \sum_{p \in \text{neighbors}(s)} (F_D(p) - F_C(p))
$$

* *Interpretation:* Samples belonging to the disease group will yield a strongly positive DCS, while control samples will hover around zero or negative.

In [6]:
graph_summary_path = "../../EdgeAssignmentTask/gumbel_softmax/results/no_label_homophily/adni/AD_KG/ecdf_5/Summary_adni_Protein.csv"

def get_protein_degrees(graph_summary_path, mask):
    df_graph_summary = pd.read_csv(graph_summary_path, index_col=0)
    df_graph_summary['ast_linked_nodes'] = df_graph_summary['pos_linked_nodes'].apply(ast.literal_eval) + df_graph_summary['neg_linked_nodes'].apply(ast.literal_eval)
    all_proteins_degree = get_common_proteins(df_graph_summary['ast_linked_nodes'].to_dict())


    # calculate disease/control protein degrees only from training samples
    mask = pd.Series(mask, index=df_graph_summary.index)
    train_df = df_graph_summary.loc[mask]
    
    control_graph_dict = train_df[train_df['label']==0]['ast_linked_nodes'].to_dict()
    disease_graph_dict = train_df[train_df['label']==1]['ast_linked_nodes'].to_dict()

    control_proteins_degree = get_common_proteins(control_graph_dict)
    disease_proteins_degree = get_common_proteins(disease_graph_dict)

    # compute disease/control sample numbers
    total_disease_samples = len(train_df[train_df['label']==1])
    total_control_samples = len(train_df[train_df['label']==0])
    # compute weighted adjacency vector for samples
    protein_weights = {}
    for p in all_proteins_degree:
        d_count = disease_proteins_degree.get(p,0)
        c_count = control_proteins_degree.get(p,0)

        freq_d = (d_count + 1) / (total_disease_samples + 1)
        freq_c = (c_count + 1) / (total_control_samples + 1)
        protein_weights[p] = freq_d/freq_c

    return all_proteins_degree, control_proteins_degree, disease_proteins_degree, protein_weights

In [ ]:
def compute_topological_features(G, graph_summary_path, mask, label_homophily=False):
    """
    Compute topological features for patient nodes in the graph.
    
    Args:
        G: NetworkX graph object
        graph_summary_path:str
        mask:list|np.array|tensor
        node_type:str
        label_homophily:bool
        
    Returns:
        dict: Dictionary containing topological features for each node
    """
    print("Computing topological features...")
    
    all_proteins_degree, control_proteins_degree, disease_proteins_degree, protein_weights = get_protein_degrees(graph_summary_path, mask)
    
    top_50_disease_proteins = list(disease_proteins_degree.keys())[:50]
    
    df_graph_summary = pd.read_csv(graph_summary_path, index_col=0)
    mask = pd.Series(mask, index=df_graph_summary.index)
    train_df = df_graph_summary.loc[mask]
    train_nodes = list(train_df.index)

    # compute features
    degrees = []
    mean_neighbor_degree = []
    max_neighbor_degree = []
    min_neighbor_degree = []
    disease_hub_overlap_score = []
    dcs=[]
    avg_protein_weights = []

    sample_nodes = nodes = [node for node,attr in G.nodes(data=True) if attr.get('label')=='Patient']
    # Centrality measures
    all_degree_centrality = nx.degree_centrality(G)
    degree_centrality = np.array([all_degree_centrality[node] for node in sample_nodes])

    for sample in sample_nodes:
        # Get all protein neighbors for this sample
        neighbors = list(G.neighbors(sample))
        degrees.append(len(neighbors))
        
        # Extract the pre-calculated degrees of the connected protein neighbors
        neighbor_degrees = [all_proteins_degree.get(p, 0) for p in neighbors]
        #print(neighbor_degrees)
        
        # mean_neighbor_degree
        mean_deg = np.mean(neighbor_degrees)
        mean_neighbor_degree.append(mean_deg)
        
        # max_neighbor_degree
        max_deg = np.max(neighbor_degrees)
        max_neighbor_degree.append(max_deg)

        # min_neighbor_degree
        min_deg = np.min(neighbor_degrees)
        min_neighbor_degree.append(min_deg)
        
        # Disease Hub Overlap Score
        overlap_count = sum(1 for p in neighbors if p in top_50_disease_proteins)
        overlap_score = overlap_count / len(neighbors)
        disease_hub_overlap_score.append(overlap_score)
        #print(overlap_score)

        # compute DCS(Differential Connectivity Score)
        fd = np.sum([disease_proteins_degree.get(p,0) for p in neighbors])
        fc = np.sum([control_proteins_degree.get(p,0) for p in neighbors])
        dcs.append(fd-fc)

        # average protein frequency ratio weight
        pr_weight = np.average([protein_weights.get(p,0) for p in neighbors])
        avg_protein_weights.append(pr_weight)

    features = {
            'degrees': np.array(degrees),
            'mean_neighborhood_degrees':np.array(mean_neighbor_degree),
            'max_neighborhood_degrees':np.array(max_neighbor_degree),
            'min_neighborhood_degrees':np.array(min_neighbor_degree),
            'disease_hub_overlap_score':np.array(disease_hub_overlap_score),
            'differential_connectivity_score':np.array(dcs),
            'avg_protein_disease_ratio':np.array(avg_protein_weights),
        }
    
    # Two-hop agreement
    if label_homophily:
        two_hop_agreement = []
        for n in tqdm(nodes, desc="Computing two-hop agreement"):
            d2 = set(nx.single_source_shortest_path_length(G, n, cutoff=2).keys())
            d22 = list(d2)
            for neighbor in d2:
                if neighbor == n or G.nodes()[neighbor].get('label')!= 'Patient' or neighbor not in train_nodes:
                    d22.remove(neighbor)
            # print(d22)
            if d22:
                lbls = [G.nodes[m]['y'] for m in d22]
                two_hop_agreement.append(lbls.count(G.nodes[n]['y']) / len(lbls))
            else:
                two_hop_agreement.append(0.0)
        two_hop_agreement = np.array(two_hop_agreement)
        features = features | {'two_hop_agreement': two_hop_agreement}

    return features

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split


df_graph_summary = pd.read_csv(graph_summary_path, index_col=0)
df_graph_summary['ast_linked_nodes'] = df_graph_summary['pos_linked_nodes'].apply(ast.literal_eval) + df_graph_summary['neg_linked_nodes'].apply(ast.literal_eval)

num_patients = len(df_graph_summary)
y_all = df_graph_summary['label'].to_list()
outer_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42,)

for outer_fold, (trainval_idx, test_idx) in enumerate(
        outer_skf.split(np.zeros(num_patients), y_all)
    ):

    data['Patient'].train_mask = torch.zeros(num_patients, dtype=torch.bool)
    data['Patient'].train_mask[trainval_idx] = True
    data['Patient'].test_mask = torch.zeros(num_patients, dtype=torch.bool)
    data['Patient'].test_mask[test_idx] = True

    nt_topo_features = compute_topological_features(G, 
                                                    graph_summary_path, 
                                                    data['Patient'].train_mask, 
                                                    label_homophily=True)
    nt_topo = np.vstack([nt_topo_features[name] for name in nt_topo_features.keys()]).T
    data['Patient'].topo_features = torch.tensor(nt_topo,dtype=torch.float)

    break

Computing topological features...


In [ ]:
mask = pd.Series(mask, index=df_graph_summary.index)
train_df = df_graph_summary.loc[mask]

In [86]:
data['Patient'].topo_features.size()

torch.Size([744, 7])

In [88]:
nt_topo_features.keys()

dict_keys(['degrees', 'mean_neighborhood_degrees', 'max_neighborhood_degrees', 'min_neighborhood_degrees', 'disease_hub_overlap_score', 'differential_connectivity_score', 'avg_protein_disease_ratio'])